# Prepare Demo 

Notebook to run to prepare data ready for demo.

In [10]:
import polars as pl
from pathlib import Path
import duckdb

from data_importers import WorldBankDataImporter
importer = WorldBankDataImporter()

In [11]:
DUCKDB_PATH = Path("../../databases") / "world_bank.db"
DUCKDB_PATH.parent.mkdir(parents=True, exist_ok=True)

if DUCKDB_PATH.exists():
    # Remove the file, we want to start from scratch
    DUCKDB_PATH.unlink()

In [12]:
START_YEAR = 1970
END_YEAR = 2024

SELECTED_COUNTRIES = [
    'GBR', 'FRA', 'DEU', 'ALB', 'NOR', 'GRC',   #Europe & Central Asia
    'JPN', 'CHN',                               #East Asia & Pacific
    'IND', 'PAK', 'BGD',                        #South Asia
    'SAU', 'EGY', 'ISR',                        #Middle East & North Africa
    'NGA', 'ZAF', 'TZA',                        #Sub-Saharan Africa
    'USA', 'CAN', 'MEX',                        #North America
    'CUB', 'DOM', 'CRI', 'BRA', 'ARG', 'CHL',   #Latin America & Caribbean
    'AUS', 'NZL', 'PHL', 'IDN', 'VNM', 'THA'    #East Asia & Pacific
]

SELECTED_INDICATORS = [
    'SP.POP.TOTL',          # Population, total
    'NY.GDP.MKTP.CD',       # GDP (current US$)
    'SE.ADT.LITR.ZS',       # Literacy rate, adult total
    'SL.UEM.TOTL.ZS',       # Unemployment, total (% of  total labor force)
    'SP.DYN.LE00.IN',       # Life expectancy at birth, total (years)
    'EG.USE.PCAP.KG.OE',    # Energy use per capita (kg of oil equivalent)
    'GC.DOD.TOTL.GD.ZS',    # Central government debt, total (% of GDP)
    'NY.GDP.PCAP.CD',       # GDP (US$, per capita)
]

## Get List of Countries

In [13]:
countries = importer.get_countries()

INFO:root:Fetching countries page 1...


In [14]:
countries.columns

['country_code',
 'iso2_code',
 'country_name',
 'region',
 'region_id',
 'income_level',
 'capital_city',
 'longitude',
 'latitude']

In [15]:
countries = countries.filter(pl.col("region") != "Aggregates")

In [16]:
# Cast longitude and latitude to double
countries = countries.with_columns(
    pl.col("longitude").cast(pl.Float64, strict=False),
    pl.col("latitude").cast(pl.Float64, strict=False)
)

In [17]:
countries.filter(pl.col("country_code").is_in(SELECTED_COUNTRIES))

country_code,iso2_code,country_name,region,region_id,income_level,capital_city,longitude,latitude
str,str,str,str,str,str,str,f64,f64
"""ALB""","""AL""","""Albania""","""Europe & Central Asia""","""ECS""","""Upper middle income""","""Tirane""",19.8172,41.3317
"""ARG""","""AR""","""Argentina""","""Latin America & Caribbean ""","""LCN""","""Upper middle income""","""Buenos Aires""",-58.4173,-34.6118
"""AUS""","""AU""","""Australia""","""East Asia & Pacific""","""EAS""","""High income""","""Canberra""",149.129,-35.282
"""BGD""","""BD""","""Bangladesh""","""South Asia""","""SAS""","""Lower middle income""","""Dhaka""",90.4113,23.7055
"""BRA""","""BR""","""Brazil""","""Latin America & Caribbean ""","""LCN""","""Upper middle income""","""Brasilia""",-47.9292,-15.7801
…,…,…,…,…,…,…,…,…
"""THA""","""TH""","""Thailand""","""East Asia & Pacific""","""EAS""","""Upper middle income""","""Bangkok""",100.521,13.7308
"""TZA""","""TZ""","""Tanzania""","""Sub-Saharan Africa ""","""SSF""","""Lower middle income""","""Dodoma""",35.7382,-6.17486
"""USA""","""US""","""United States""","""North America""","""NAC""","""High income""","""Washington D.C.""",-77.032,38.8895


## Download indicator data

For each country, for each indicator between the years chosen.

In [18]:
data = importer.get_data(
    indicators=SELECTED_INDICATORS,
    countries=SELECTED_COUNTRIES,
    start_year=START_YEAR,
    end_year=END_YEAR
)

INFO:root:Fetching data with skip=0...
INFO:root:Fetching data with skip=1000...
INFO:root:Fetching data with skip=2000...
INFO:root:Fetching data with skip=3000...
INFO:root:Fetching data with skip=4000...
INFO:root:Fetching data with skip=5000...
INFO:root:Fetching data with skip=6000...
INFO:root:Fetching data with skip=7000...
INFO:root:Fetching data with skip=8000...
INFO:root:Fetching data with skip=9000...


In [19]:
data

country_code,year,indicator_code,value
str,i64,str,f64
"""VNM""",1999,"""WB_WDI_EG_USE_PCAP_KG_OE""",358.075426
"""VNM""",2001,"""WB_WDI_EG_USE_PCAP_KG_OE""",393.081627
"""VNM""",2008,"""WB_WDI_EG_USE_PCAP_KG_OE""",572.099542
"""VNM""",2007,"""WB_WDI_EG_USE_PCAP_KG_OE""",544.673183
"""USA""",2000,"""WB_WDI_EG_USE_PCAP_KG_OE""",8055.145489
…,…,…,…
"""MEX""",1970,"""WB_WDI_SP_POP_TOTL""",5.0814953e7
"""NZL""",1970,"""WB_WDI_SP_POP_TOTL""",2.8107e6
"""NGA""",1970,"""WB_WDI_SP_POP_TOTL""",5.5893838e7


## Capture indicator metadata

In [20]:
indicators = importer.get_indicator_metadata(indicators=SELECTED_INDICATORS)
indicators

INFO:root:Fetching metadata for 8 indicators...


indicator_code,indicator_name,aggregation_method,definition_long,statistical_concept,methodology,limitation,relevance
str,str,str,str,str,str,str,str
"""WB_WDI_NY_GDP_PCAP_CD""","""GDP per capita (current US$)""","""Weighted average""","""Gross domestic product is the …","""The conceptual elements of the…","""National accounts are compiled…",null,"""This indicator is related to t…"
"""WB_WDI_GC_DOD_TOTL_GD_ZS""","""Central government debt, total…","""Weighted average""","""Debt is the entire stock of di…","""Government Financial Statistic…","""Government Finance statistics …","""For most countries central gov…","""This indicator is related to G…"
"""WB_WDI_SP_DYN_LE00_IN""","""Life expectancy at birth, tota…","""Weighted average""","""Life expectancy at birth indic…","""Life expectancy at birth used …","""Life expectancy at birth is de…","""Annual data series from United…","""Mortality rates for different …"
"""WB_WDI_SL_UEM_TOTL_ZS""","""Unemployment, total (% of tota…","""Weighted average""","""Unemployment refers to the sha…","""The unemployed comprise all pe…","""The unemployment rate is calcu…","""While the unemployment rate ma…","""The unemployment rate is a use…"
"""WB_WDI_SP_POP_TOTL""","""Population, total""","""Sum""","""Total population is based on t…","""Estimates of total population …","""Population estimates are usual…","""Current population estimates f…","""Increases in human population,…"
"""WB_WDI_SE_ADT_LITR_ZS""","""Literacy rate, adult total (% …","""Weighted average""","""Adult literacy rate is the per…","""Literacy statistics for most c…","""The indicator is calculated by…","""In practice, literacy is diffi…","""Literacy rate is an outcome in…"
"""WB_WDI_EG_USE_PCAP_KG_OE""","""Energy use (kg of oil equivale…","""Weighted average""","""Energy use refers to use of pr…",null,"""Total energy use refers to the…","""The IEA makes these estimates …","""In developing economies growth…"
"""WB_WDI_NY_GDP_MKTP_CD""","""GDP (current US$)""","""Gap-filled total""","""Gross domestic product is the …","""The conceptual elements of the…","""National accounts are compiled…","""Gross domestic product (GDP), …","""This indicator is related to t…"


## DuckDB

In [21]:
with duckdb.connect(DUCKDB_PATH) as db:

    db.sql("CREATE SCHEMA IF NOT EXISTS silver;")

    db.sql(
        f"""
        CREATE OR REPLACE TABLE silver.countries AS
        SELECT *
        FROM countries;
        """
    )

    db.sql(
        f"""
        CREATE OR REPLACE TABLE silver.data AS
        SELECT *
        FROM data;
        """
    )

    db.sql(
        f"""
        CREATE OR REPLACE TABLE silver.indicators AS
        SELECT *
        FROM indicators;
        """
    )

db.close()